# Generated collection worker 1/3

Generated from `04_collect_verifier_pairs.ipynb`; edit the canonical notebook, not this copy.


# 04 — Deterministic-replay verifier collection

Collect up to 300 unique mid-rollout states × 4 candidates without simulator snapshots. Each branch reaches its state by replaying an identical fixed action prefix. Resumable and shardable; v1/v2 data is preserved.

## 1. Environment setup

In [ ]:
import os, subprocess, sys
try:
    from google.colab import userdata
    for key in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "HF_TOKEN", "WANDB_API_KEY"):
        value = userdata.get(key)
        if value:
            os.environ[key] = value
    repo_dir = "/content/cs159-sp26"
    gh_pat = userdata.get("GH_PAT")
    repo_url = f"https://{gh_pat}@github.com/ArjunS07/cs159-sp26.git"
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        subprocess.run(["git", "clone", "--branch", "main", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", "main"], check=True)
except ImportError:
    repo_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "pnp-vla" else os.getcwd()

package_dir = os.path.join(repo_dir, "pnp-vla")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", package_dir + "[sim,analysis]"], check=True)
if package_dir not in sys.path:
    sys.path.insert(0, package_dir)
import pnp
print("Loaded pnp from:", pnp.__file__)

## 2. Load policy, store, and benchmark episode manifests

In [ ]:
import json
from tqdm.auto import tqdm
from pnp import libero_env, libero_pro, models
from pnp.experiments import _prepare_libero_pro_episodes
from pnp.store import SupabaseStore
from pnp.verifier import *

benchmark_dict = libero_env.init_libero_benchmark()
libero_pro.patch_torch_load()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()

suites = ("libero_spatial", "libero_object", "libero_goal", "libero_10")
tasks = [(suite, task) for suite in suites for task in range(benchmark_dict[suite]().n_tasks)]
standard = libero_env.build_final_episodes(benchmark_dict, tasks=tasks)
for ep in standard: ep["benchmark"] = "libero"
pro = _prepare_libero_pro_episodes()
for ep in pro: ep["benchmark"] = "libero_pro"
episode_lookup = {(e["benchmark"], e["suite"], e["task_idx"], e.get("ep_idx", e.get("episode_idx"))): e
                  for e in standard + pro}
print(len(standard), len(pro), len(episode_lookup))

## 3. Configure workers and build the fixed uncertainty manifest

In [ ]:
COLLECTION_EXPERIMENT = "verifier-clean-pairs-v3"
TARGETS = {"libero": 120, "libero_pro": 180}
CANDIDATE_COUNT = 4
PREFIX_LENGTH = 10
SHARD_COUNT = 3   # Generated worker count.
SHARD_INDEX = 1   # Generated worker index.
assert 0 <= SHARD_INDEX < SHARD_COUNT

def pages(table, columns, configure):
    rows=[]; start=0
    while True:
        q = configure(store.client.table(table).select(columns)).range(start, start+999)
        batch = q.execute().data or []; rows += batch
        if len(batch) < 1000: return rows
        start += 1000

experiments = ("libero-hybrid-schedules-k3-v1", "libero-pro-canonical-core-k3-v1")
rollouts=[]
for experiment in experiments:
    rollouts += pages("rollouts", "rollout_id,benchmark,suite,task_idx,episode_idx,success", lambda q, e=experiment:
        q.eq("experiment", e).eq("method", "pnp_uncertainty_only").eq("status", "completed"))
ids = {r["rollout_id"] for r in rollouts}
euler = []
id_list = sorted(ids)
for start in range(0, len(id_list), 100):
    batch_ids = id_list[start:start+100]
    euler += pages("pnp_euler_steps", "rollout_id,chunk_idx,u_mean",
                   lambda q, batch_ids=batch_ids: q.in_("rollout_id", batch_ids))
manifest = build_stratified_manifest(rollouts, euler, TARGETS)
TOTAL_GROUPS = len(manifest)
if TOTAL_GROUPS < sum(TARGETS.values()):
    print(f"eligible unique-state capacity: {TOTAL_GROUPS}/{sum(TARGETS.values())}; "
          "collecting all available groups")
manifest = manifest[SHARD_INDEX::SHARD_COUNT]
print({k: sum(r["benchmark"] == k for r in manifest) for k in ("libero", "libero_pro")})
print({k: sum(r["uncertainty_stratum"] == k for r in manifest) for k in ("low", "mid", "high")})

## 4. Dry-run identity and schema checks

In [ ]:
missing = [row for row in manifest if (row["benchmark"], row["suite"], row["task_idx"], row["episode_idx"]) not in episode_lookup]
assert not missing, missing[:3]
store.client.table("verifier_candidate_groups").select("candidate_group_id").limit(1).execute()
print("manifest identities and verifier tables are ready")

## 5. Collect deterministic-replay four-candidate groups

In [ ]:
existing_group_rows = pages(
    "verifier_candidate_groups", "candidate_group_id",
    lambda q: q.eq("experiment", COLLECTION_EXPERIMENT))
existing_group_ids = {row["candidate_group_id"] for row in existing_group_rows}
all_candidate_ids = pages(
    "verifier_candidates", "candidate_group_id",
    lambda q: q)
candidate_counts = {}
for row in all_candidate_ids:
    gid = row["candidate_group_id"]
    if gid in existing_group_ids:
        candidate_counts[gid] = candidate_counts.get(gid, 0) + 1
# A crash can leave a group row with fewer than four candidates. Recollect it.
existing = {gid for gid in existing_group_ids
            if candidate_counts.get(gid, 0) == CANDIDATE_COUNT}
print({"complete_existing_groups": len(existing),
       "partial_groups_to_repair": len(existing_group_ids - existing)})
store.start_run("verifier_pair_collection", "libero+libero_pro", COLLECTION_EXPERIMENT,
                config={"groups": TOTAL_GROUPS,
                        "outcomes": TOTAL_GROUPS * CANDIDATE_COUNT,
                        "candidate_count": CANDIDATE_COUNT, "prefix_length": PREFIX_LENGTH,
                        "shard_count": SHARD_COUNT, "shard_index": SHARD_INDEX})
completed = 0
for item in tqdm(manifest, desc="candidate groups"):
    ep = episode_lookup[(item["benchmark"], item["suite"], item["task_idx"], item["episode_idx"])]
    expected_id = candidate_group_id(item["benchmark"], item["suite"], item["task_idx"],
                                     item["episode_idx"], item["chunk_idx"],
                                     namespace=COLLECTION_EXPERIMENT)
    if expected_id in existing:
        continue
    env = libero_env.make_env(ep["bddl_path"])
    try:
        try:
            pair = collect_replay_candidate_group(
                env, ep, policy, preprocess, postprocess, device,
                chunk_idx=item["chunk_idx"], uncertainty_stratum=item["uncertainty_stratum"],
                prefix_length=PREFIX_LENGTH,
                candidate_count=CANDIDATE_COUNT, experiment=COLLECTION_EXPERIMENT)
        except Exception as error:
            print("replay group skipped:", type(error).__name__, error)
            pair = None
        if pair is None:
            continue
        group, candidates = pair
        store.register_candidate_group(group, candidates)
        existing.add(group["candidate_group_id"]); completed += len(candidates)
    finally:
        env.close()
store.finish_run(n_rollouts=completed)
print("new outcomes:", completed, "total groups:", len(existing))

## 6. Integrity and outcome-balance report

In [ ]:
groups = pages(
    "verifier_candidate_groups", "*",
    lambda q: q.eq("experiment", COLLECTION_EXPERIMENT))
candidates = pages(
    "verifier_candidates", "candidate_id,candidate_group_id,candidate_kind,success",
    lambda q: q)
group_ids = {g["candidate_group_id"] for g in groups}
candidates = [c for c in candidates if c["candidate_group_id"] in group_ids]
by_group = {}
for candidate in candidates:
    by_group.setdefault(candidate["candidate_group_id"], []).append(candidate)
complete = {gid for gid in group_ids if len(by_group.get(gid, [])) == CANDIDATE_COUNT}
print({"groups": len(groups), "outcomes": len(candidates),
       "complete_groups": len(complete),
       "partial_groups": len(group_ids - complete),
       "successes": sum(c["success"] for c in candidates),
       "failures": sum(not c["success"] for c in candidates),
       "discordant_groups": sum(len({c["success"] for c in by_group.get(gid, [])}) == 2
                                for gid in complete),
       "replay_groups": sum(g["pairing_mode"] == "deterministic_replay" for g in groups),
       "validated_groups": sum(bool((g.get("metadata_json") or {}).get("replay_validated"))
                               for g in groups)})